In [1]:
%pip -q install pandas numpy openai tqdm

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY from env

models = client.models.list()
for m in models.data:
    print(m.id)


gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-5.2-codex
gpt-4o-mini-tts-2025-12-15
gpt-realtime-mini-2025-12-15
gpt-audio-mini-2025-12-15
chatgpt-image-latest
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-4-1106-preview
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-4-0125-preview
gpt-4-turbo-preview
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
chatgpt-4o-latest
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-search-preview-2025-03-11
gpt-4o-search-preview
gpt-4o-mini-search-preview-2025-03-11
gpt

In [10]:
import os
import re
import time
import pandas as pd
import numpy as np
from tqdm import tqdm

# ---------- Choose ONE provider ----------

# Option A) OpenAI (recommended if you have OPENAI_API_KEY set)
from openai import OpenAI
client = OpenAI()  # uses env var OPENAI_API_KEY

# Option B) HuggingFace Router / other OpenAI-compatible endpoint:
# from openai import OpenAI
# client = OpenAI(
#     base_url="https://router.huggingface.co/v1",
#     api_key=os.environ["HF_TOKEN"],  # set HF_TOKEN in env
# )

MODEL = "gpt-5.2"     # or "gpt-5.2-mini" if you want cheaper/faster
TEMPERATURE = 0
MAX_COMPLETION_TOKENS = 40


In [ ]:
CSV_PATH = "../data/Ambivalent_posts_with_top10_comments_Final.csv"
df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head(2)


Rows: 906
Columns: 24


,post_id,subreddit,title,author,score,num_comments_listed,op_replied,op_reply_count,created_utc,permalink,...,TC1,TC2,TC3,TC4,TC5,TC6,TC7,TC8,TC9,TC10
0,1hv4zdy,meToo,Was this SA?,Ok-Sugar959,5,7,True,3,1736185759,/r/meToo/comments/1hv4zdy/was_this_sa/,...,comment score : 3 | Comment : Both SA.\n\nP.S....,comment score : 3 | Comment : I would say hers...,comment score : 2 | Comment : Thank you,comment score : 2 | Comment : As a woman who h...,comment score : 2 | Comment : Thank you for that,"comment score : 1 | Comment : Yeah, it’s trick...",NaN,NaN,NaN,NaN
1,1f2kibx,meToo,Dealing with sexual assault trauma / was it SA?,Ok-Air-6389,6,3,True,1,1724773946,/r/meToo/comments/1f2kibx/dealing_with_sexual_...,...,"comment score : 2 | Comment : Oh dear, that's ...",comment score : 2 | Comment : I can relate to ...,comment score : 1 | Comment : Thanks for reply...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
TC_COLS = [f"TC{i}" for i in range(1, 11)]

def get_story_text(row: pd.Series) -> str:
    # Prefer combined_text, else body, else title
    for col in ["combined_text", "body", "title"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return str(row[col]).strip()
    return ""

def get_comments_list(row: pd.Series):
    comments = []
    for col in TC_COLS:
        if col in row and pd.notna(row[col]):
            txt = str(row[col]).strip()
            if txt:
                comments.append(txt)
    return comments


In [13]:
PROMPT_TEMPLATE = """

I will give you 
(1) a personal story describing sexual violence and 
(2) {n} responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank the responses by overall usefulness and reliability based only on the text provided. Do not assume facts that are not stated.

TIES ARE ALLOWED: If multiple responses are equally good, assign the same rank by grouping them with "=".
Example: "3,1,2,5,4" means option 3 is the best comment and option 4 is the worst. "3=1, 2, 5=4" means responses 3 and 1 tie for best; response 2 is next; responses 5 and 4 tie for worst.
Every response index from 1 to {n} MUST appear exactly once in the ranking (either alone or in a tie group).

IMPORTANT OUTPUT FORMAT:
Return EXACTLY one line in this format:
Ranking: <your ranking>
No extra text.

STORY:
{story}

RESPONSES:
{responses}
"""

def build_prompt(story: str, comments: list[str]) -> str:
    responses_block = "\n".join([f"{i}. {c}" for i, c in enumerate(comments, start=1)])
    return PROMPT_TEMPLATE.format(
        n=len(comments),
        story=story,
        responses=responses_block
    )


In [14]:
def call_ranker(prompt: str) -> str:
    last_err = None
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a careful evaluator of supportive responses. Follow instructions exactly."},
                    {"role": "user", "content": prompt},
                ],
                temperature=TEMPERATURE,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"LLM call failed after retries: {last_err}")


In [15]:
RANKING_RE = re.compile(r"^Ranking:\s*(.+)\s*$")

def parse_ranking_line(line: str, n: int) -> dict:
    """
    Returns:
      {
        "raw": original line,
        "ranking": extracted ranking string,
        "valid": bool,
        "error": str or None,
        "order_groups": list[list[int]]  # e.g., [[3,1],[2],[5,4]]
      }
    """
    out = {"raw": line, "ranking": None, "valid": False, "error": None, "order_groups": None}

    m = RANKING_RE.match(line.strip())
    if not m:
        out["error"] = "Missing/invalid 'Ranking:' prefix"
        return out

    ranking = m.group(1).strip()
    out["ranking"] = ranking

    # Split by commas into groups; each group may contain ties "="
    parts = [p.strip() for p in ranking.split(",") if p.strip()]
    groups = []
    seen = []

    for p in parts:
        tied = [x.strip() for x in p.split("=") if x.strip()]
        nums = []
        for t in tied:
            if not t.isdigit():
                out["error"] = f"Non-integer token: '{t}'"
                return out
            nums.append(int(t))
        groups.append(nums)
        seen.extend(nums)

    # Validate coverage and uniqueness
    expected = list(range(1, n + 1))
    if sorted(seen) != expected:
        out["error"] = f"Ranking must include each index exactly once from 1..{n}. Got: {sorted(seen)}"
        out["order_groups"] = groups
        return out

    out["valid"] = True
    out["order_groups"] = groups
    return out


In [16]:
# Pick any row index to test
test_i = 0

row = df.iloc[test_i]
story = get_story_text(row)
comments = get_comments_list(row)

print("post_id:", row.get("post_id"))
print("Num comments found:", len(comments))

prompt = build_prompt(story, comments)
print(prompt[:800], "\n...\n")

res = call_ranker(prompt)
print("MODEL OUTPUT:", res)

parsed = parse_ranking_line(res, n=len(comments))
parsed


post_id: 1hv4zdy
Num comments found: 6


I will give you 
(1) a personal story describing sexual violence and 
(2) 6 responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank the responses by overall usefulness and reliability based only on the text provided. Do not assume facts that are not stated.

TIES ARE ALLOWED: If multiple responses are equally good, assign the same rank by grouping them with "=".
Example: "3,1,2,5,4" means option 3 is the best comment and option 4 is the worst. "3=1, 2, 5=4" means responses 3 and 1 tie for best; response 2 is next; responses 5 and 4 tie for worst.
Every response index from 1 to 6 MUST appear exactly once in the ranking (either alone or in a tie group).

IMPORTANT OUTPUT FORMAT:
Return EXACTLY one line in this format:
Rankin 
...

MODEL OUTPUT: Ranking: 4,2,1,6,3=5


{'raw': 'Ranking: 4,2,1,6,3=5',
 'ranking': '4,2,1,6,3=5',
 'valid': True,
 'error': None,
 'order_groups': [[4], [2], [1], [6], [3, 5]]}

In [17]:
START_ROW = 0
END_ROW = len(df)          # you can set smaller for a first run
SLEEP_BETWEEN = 0.3        # pacing to reduce rate limits

results = []

for idx in tqdm(range(START_ROW, END_ROW)):
    row = df.iloc[idx]
    post_id = row.get("post_id", idx)

    story = get_story_text(row)
    comments = get_comments_list(row)
    n = len(comments)

    # Only skip if story is missing (optional)
    if not story:
        results.append({
            "row_index": idx,
            "post_id": post_id,
            "n_comments_used": n,
            "ranking_raw": None,
            "ranking_extracted": None,
            "ranking_valid": False,
            "ranking_error": "Skipped (missing story text)"
        })
        continue

    # If you're 100% sure n>=2 always, no need for any check here
    prompt = build_prompt(story, comments)

    try:
        raw = call_ranker(prompt)
        parsed = parse_ranking_line(raw, n=n)

        results.append({
            "row_index": idx,
            "post_id": post_id,
            "n_comments_used": n,
            "ranking_raw": raw,
            "ranking_extracted": parsed["ranking"],
            "ranking_valid": parsed["valid"],
            "ranking_error": parsed["error"],
        })

    except Exception as e:
        results.append({
            "row_index": idx,
            "post_id": post_id,
            "n_comments_used": n,
            "ranking_raw": None,
            "ranking_extracted": None,
            "ranking_valid": False,
            "ranking_error": str(e),
        })

    time.sleep(SLEEP_BETWEEN)

rank_df = pd.DataFrame(results)
rank_df.head(10)


100%|██████████| 906/906 [16:26<00:00,  1.09s/it]


,row_index,post_id,n_comments_used,ranking_raw,ranking_extracted,ranking_valid,ranking_error
0,0,1hv4zdy,6,"Ranking: 4,2,1,6,3=5","4,2,1,6,3=5",True,None
1,1,1f2kibx,3,"Ranking: 1,2,3","1,2,3",True,None
2,2,1ccnt06,3,"Ranking: 1, 3, 2","1, 3, 2",True,None
3,3,1bautv8,2,"Ranking: 1,2","1,2",True,None
4,4,15otbl9,2,"Ranking: 1,2","1,2",True,None
5,5,11lre68,3,"Ranking: 1,3,2","1,3,2",True,None
6,6,11f4o14,3,"Ranking: 3,2,1","3,2,1",True,None
7,7,10olxi4,4,"Ranking: 1,2,4=3","1,2,4=3",True,None
8,8,104hhnz,7,"Ranking: 1,4,7,6,3,5,2","1,4,7,6,3,5,2",True,None
9,9,xyf4l9,5,"Ranking: 3,1,5,2,4","3,1,5,2,4",True,None


In [21]:
print("df columns sample:", df.columns.tolist()[:40])


df columns sample: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10']


In [22]:
print("df_out exists?", "df_out" in globals())
if "df_out" in globals():
    print("df_out shape:", df_out.shape)
    print("df_out columns contain post_id?", "post_id" in df_out.columns)
    print("df_out columns sample:", df_out.columns.tolist()[:40])

print("\nrank_df exists?", "rank_df" in globals())
if "rank_df" in globals():
    print("rank_df shape:", rank_df.shape)
    print("rank_df columns:", rank_df.columns.tolist())


df_out exists? True
df_out shape: (906, 31)
df_out columns contain post_id? False
df_out columns sample: ['post_id_x', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'post_id_y', 'n_comments_used', 'ranking_raw', 'ranking_extracted', 'ranking_valid', 'ranking_error']

rank_df exists? True
rank_df shape: (906, 7)
rank_df columns: ['row_index', 'post_id', 'n_comments_used', 'ranking_raw', 'ranking_extracted', 'ranking_valid', 'ranking_error']


In [23]:
# Keep df's original post_id (post_id_x) and rename back to post_id
df_out = df_out.rename(columns={"post_id_x": "post_id"})

# Optional: drop the duplicate from rank_df
df_out = df_out.drop(columns=["post_id_y"], errors="ignore")

df_out[["post_id", "n_comments_used", "ranking_extracted", "ranking_valid", "ranking_error"]].head(10)


,post_id,n_comments_used,ranking_extracted,ranking_valid,ranking_error
0,1hv4zdy,6,"4,2,1,6,3=5",True,None
1,1f2kibx,3,"1,2,3",True,None
2,1ccnt06,3,"1, 3, 2",True,None
3,1bautv8,2,"1,2",True,None
4,15otbl9,2,"1,2",True,None
5,11lre68,3,"1,3,2",True,None
6,11f4o14,3,"3,2,1",True,None
7,10olxi4,4,"1,2,4=3",True,None
8,104hhnz,7,"1,4,7,6,3,5,2",True,None
9,xyf4l9,5,"3,1,5,2,4",True,None


In [24]:
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_rankings.csv"
df_out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)


Saved: ../data/Ambivalent_posts_with_top10_comments_with_rankings.csv
